# Assignment 4 - Quality Check Metrics

This notebook implements three common information retrieval evaluation metrics:

1. **Average Precision (AP)** - Measures precision at each relevant document position, averaged
2. **Reciprocal Rank (RR)** - The inverse of the rank position of the first relevant document
3. **NDCG (Normalized Discounted Cumulative Gain)** - Measures ranking quality using graded relevance

## Task Overview
- **5 queries**, each returning **3 answers**
- Randomly select the order of answers for each query
- For each query, select the order of possible indexes (including 0 = not relevant)
- Calculate and print metrics for each query
- Display average results across all 5 queries

---
## Setup and Imports

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Set random seed for reproducibility
np.random.seed(42)

---
## Step A: Input Preparation

We simulate a search system returning 3 answers per query. Each answer has a **relevance score**:
- `0` = Not relevant
- `1` = Relevant
- `2` = Highly relevant (used for NDCG)

For each of the 5 queries, we randomly shuffle the order of the 3 answers to simulate different ranking scenarios.

In [2]:
# Define relevance judgments for 5 queries
# Each query has 3 answers with known relevance (ground truth)
# Format: [relevance_of_answer_1, relevance_of_answer_2, relevance_of_answer_3]
# Values: 0 = not relevant, 1 = relevant, 2 = highly relevant

RELEVANCE_JUDGMENTS = {
    "Query 1": [2, 1, 0],  # One highly relevant, one relevant, one not relevant
    "Query 2": [1, 0, 0],  # One relevant, two not relevant
    "Query 3": [0, 0, 0],  # All not relevant (worst case)
    "Query 4": [2, 2, 1],  # All relevant (best case)
    "Query 5": [1, 1, 0],  # Two relevant, one not relevant
}

def shuffle_rankings(relevance_list):
    """
    Randomly shuffle the order of answers to simulate different ranking results.
    Returns the shuffled relevance scores and the permutation indices.
    """
    indices = np.array([1, 2, 3])  # Original positions (1-indexed for display)
    relevance = np.array(relevance_list)
    
    # Random permutation
    perm = np.random.permutation(len(relevance))
    shuffled_relevance = relevance[perm]
    shuffled_indices = indices[perm]
    
    return shuffled_relevance, shuffled_indices

# Prepare input: shuffle answers for each query
print("=" * 60)
print("STEP A: Input Preparation - Random Answer Orderings")
print("=" * 60)

query_results = {}

for query_name, original_relevance in RELEVANCE_JUDGMENTS.items():
    shuffled_rel, shuffled_idx = shuffle_rankings(original_relevance)
    query_results[query_name] = {
        'original': original_relevance,
        'shuffled_relevance': shuffled_rel,
        'shuffled_indices': shuffled_idx
    }
    
    print(f"\n{query_name}:")
    print(f"  Original order: {original_relevance}")
    print(f"  Shuffled order: {shuffled_rel.tolist()}")
    print(f"  Answer positions: {shuffled_idx.tolist()}")

STEP A: Input Preparation - Random Answer Orderings

Query 1:
  Original order: [2, 1, 0]
  Shuffled order: [2, 1, 0]
  Answer positions: [1, 2, 3]

Query 2:
  Original order: [1, 0, 0]
  Shuffled order: [0, 0, 1]
  Answer positions: [2, 3, 1]

Query 3:
  Original order: [0, 0, 0]
  Shuffled order: [0, 0, 0]
  Answer positions: [1, 2, 3]

Query 4:
  Original order: [2, 2, 1]
  Shuffled order: [2, 1, 2]
  Answer positions: [2, 3, 1]

Query 5:
  Original order: [1, 1, 0]
  Shuffled order: [1, 1, 0]
  Answer positions: [1, 2, 3]


---
## Step B: Calculate Metrics for Each Query

### Metric Definitions:

**1. Reciprocal Rank (RR)**
$$RR = \frac{1}{rank_{first\_relevant}}$$
- If no relevant document exists, RR = 0

**2. Average Precision (AP)**
$$AP = \frac{\sum_{k=1}^{n} P(k) \times rel(k)}{\text{total relevant documents}}$$
- P(k) = precision at position k
- rel(k) = 1 if document at position k is relevant, 0 otherwise

**3. NDCG (Normalized Discounted Cumulative Gain)**
$$DCG = \sum_{i=1}^{n} \frac{2^{rel_i} - 1}{\log_2(i + 1)}$$
$$NDCG = \frac{DCG}{IDCG}$$
- IDCG = DCG of ideal (perfectly sorted) ranking

In [3]:
def calculate_reciprocal_rank(relevance_scores):
    """
    Calculate Reciprocal Rank (RR).
    RR = 1 / rank of first relevant document.
    Returns 0 if no relevant document found.
    """
    for i, rel in enumerate(relevance_scores):
        if rel > 0:  # Found a relevant document
            return 1.0 / (i + 1)  # Rank is 1-indexed
    return 0.0


def calculate_average_precision(relevance_scores):
    """
    Calculate Average Precision (AP) for binary relevance.
    AP = sum of (precision at k * rel(k)) / total relevant documents
    """
    # Convert to binary relevance (relevant vs not relevant)
    binary_rel = [1 if r > 0 else 0 for r in relevance_scores]
    total_relevant = sum(binary_rel)
    
    if total_relevant == 0:
        return 0.0
    
    precision_sum = 0.0
    num_relevant_found = 0
    
    for i, rel in enumerate(binary_rel):
        if rel == 1:
            num_relevant_found += 1
            precision_at_k = num_relevant_found / (i + 1)  # Precision at position k
            precision_sum += precision_at_k
    
    return precision_sum / total_relevant


def calculate_dcg(relevance_scores):
    """
    Calculate Discounted Cumulative Gain (DCG).
    DCG = sum of (2^rel_i - 1) / log2(i + 1)
    """
    dcg = 0.0
    for i, rel in enumerate(relevance_scores):
        # i + 2 because position is 1-indexed and formula uses (i + 1) in log
        dcg += (2 ** rel - 1) / np.log2(i + 2)
    return dcg


def calculate_ndcg(relevance_scores):
    """
    Calculate Normalized Discounted Cumulative Gain (NDCG).
    NDCG = DCG / IDCG (ideal DCG)
    """
    dcg = calculate_dcg(relevance_scores)
    
    # Ideal ranking: sort relevance scores in descending order
    ideal_scores = sorted(relevance_scores, reverse=True)
    idcg = calculate_dcg(ideal_scores)
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg


# Calculate all metrics for each query
print("\n" + "=" * 60)
print("STEP B: Calculate Metrics for Each Query")
print("=" * 60)

metrics_results = []

for query_name, data in query_results.items():
    rel_scores = data['shuffled_relevance']
    
    rr = calculate_reciprocal_rank(rel_scores)
    ap = calculate_average_precision(rel_scores)
    ndcg = calculate_ndcg(rel_scores)
    
    metrics_results.append({
        'query': query_name,
        'relevance_order': rel_scores.tolist(),
        'reciprocal_rank': rr,
        'average_precision': ap,
        'ndcg': ndcg
    })
    
    print(f"\n{query_name}:")
    print(f"  Relevance order: {rel_scores.tolist()}")
    print(f"  Reciprocal Rank (RR):  {rr:.4f}")
    print(f"  Average Precision (AP): {ap:.4f}")
    print(f"  NDCG:                   {ndcg:.4f}")


STEP B: Calculate Metrics for Each Query

Query 1:
  Relevance order: [2, 1, 0]
  Reciprocal Rank (RR):  1.0000
  Average Precision (AP): 1.0000
  NDCG:                   1.0000

Query 2:
  Relevance order: [0, 0, 1]
  Reciprocal Rank (RR):  0.3333
  Average Precision (AP): 0.3333
  NDCG:                   0.5000

Query 3:
  Relevance order: [0, 0, 0]
  Reciprocal Rank (RR):  0.0000
  Average Precision (AP): 0.0000
  NDCG:                   0.0000

Query 4:
  Relevance order: [2, 1, 2]
  Reciprocal Rank (RR):  1.0000
  Average Precision (AP): 1.0000
  NDCG:                   0.9514

Query 5:
  Relevance order: [1, 1, 0]
  Reciprocal Rank (RR):  1.0000
  Average Precision (AP): 1.0000
  NDCG:                   1.0000


---
## Step C: Display Average Results

Compute and display the mean of each metric across all 5 queries.

In [4]:
# Convert results to DataFrame for easy display
df_results = pd.DataFrame(metrics_results)

# Calculate averages
avg_rr = df_results['reciprocal_rank'].mean()
avg_ap = df_results['average_precision'].mean()
avg_ndcg = df_results['ndcg'].mean()

print("\n" + "=" * 60)
print("STEP C: Average Results Across All 5 Queries")
print("=" * 60)

# Display individual query results in a table
print("\nIndividual Query Results:")
display(df_results[['query', 'relevance_order', 'reciprocal_rank', 'average_precision', 'ndcg']])

# Display summary averages
print("\n" + "-" * 60)
print("SUMMARY - Average Metrics:")
print("-" * 60)
print(f"  Mean Reciprocal Rank (MRR):     {avg_rr:.4f}")
print(f"  Mean Average Precision (MAP):   {avg_ap:.4f}")
print(f"  Mean NDCG:                      {avg_ndcg:.4f}")
print("-" * 60)

# Create summary DataFrame
df_summary = pd.DataFrame({
    'Metric': ['MRR (Mean Reciprocal Rank)', 
               'MAP (Mean Average Precision)', 
               'Mean NDCG'],
    'Value': [avg_rr, avg_ap, avg_ndcg]
})

print("\nSummary Table:")
display(df_summary)


STEP C: Average Results Across All 5 Queries

Individual Query Results:


,query,relevance_order,reciprocal_rank,average_precision,ndcg
0,Query 1,"[2, 1, 0]",1.000000,1.000000,1.000000
1,Query 2,"[0, 0, 1]",0.333333,0.333333,0.500000
2,Query 3,"[0, 0, 0]",0.000000,0.000000,0.000000
3,Query 4,"[2, 1, 2]",1.000000,1.000000,0.951443
4,Query 5,"[1, 1, 0]",1.000000,1.000000,1.000000



------------------------------------------------------------
SUMMARY - Average Metrics:
------------------------------------------------------------
  Mean Reciprocal Rank (MRR):     0.6667
  Mean Average Precision (MAP):   0.6667
  Mean NDCG:                      0.6903
------------------------------------------------------------

Summary Table:


,Metric,Value
0,MRR (Mean Reciprocal Rank),0.666667
1,MAP (Mean Average Precision),0.666667
2,Mean NDCG,0.690289


---
## Summary

### What Was Done:

1. **Input Preparation (Step A)**:
   - Defined 5 queries, each with 3 answers having known relevance judgments (0, 1, or 2)
   - Randomly shuffled the answer order for each query to simulate different ranking scenarios

2. **Metric Calculation (Step B)**:
   - **Reciprocal Rank (RR)**: Measures how early the first relevant answer appears
   - **Average Precision (AP)**: Measures precision across all relevant answers
   - **NDCG**: Uses graded relevance (0, 1, 2) to measure ranking quality

3. **Average Results (Step C)**:
   - Computed mean values (MRR, MAP, Mean NDCG) across all 5 queries
   - Displayed both individual query results and aggregate summary

### Key Observations:
- **Query 3** (all zeros) has metrics of 0 - no relevant answers means the system failed completely
- **Query 4** (all relevant) should have high scores across all metrics
- The **averages** give a single-number summary of overall system performance